In [5]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go


In [6]:

#gdf_rinde = gpd.read_file("datos_cosecha_300puntos.csv")
#gdf_rinde.head()

#Problema resuelto, Geopandas no lee .csv. Se lo lee con Pandas y luego se arma la geometría con gpd    
df = pd.read_csv('datos_cosecha_300puntos.csv')

gdf_rinde = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df['Longitud'], df['Latitud'])
)
# 3. Le asignamos el sistema de origen (Lat/Lon estándar)
gdf_rinde = gdf_rinde.set_crs(epsg=4326)

# 4. Ahora sí, podés reproyectar sin que explote nada:
gdf_rinde_metros = gdf_rinde.to_crs(gdf_rinde.estimate_utm_crs())

# 5. Reproyectamos
gdf_rp = gdf_rinde_metros.to_crs(epsg=32720)
gdf_rp.head()

# 6. Estimar superficie del perimetro
gdf_rp['superficie_m2'] = gdf_rp.geometry.buffer(10).area
gdf_rp.head()

FileNotFoundError: [Errno 2] No such file or directory: 'datos_cosecha_300puntos.csv'

In [ ]:
gdf_rinde['cuantiles'] = pd.qcut(gdf_rinde['Rinde_kgHa'], 5, labels=False)

gdf_rinde.plot(column='cuantiles', cmap='RdYlGn', legend=True)
plt.title('Rinde en 4 cuantiles')
plt.show()


In [ ]:
#Añadir la capa de rendimientos en un mapa interactivo con Plotly
fig = go.Figure()
fig.add_trace(go.Scattermapbox(
    lat=gdf_rinde['Latitud'],
    lon=gdf_rinde['Longitud'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=10,
        color=gdf_rinde['Rinde_kgHa'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='Rinde (kg/ha)')
    ),
    text=gdf_rinde['Rinde_kgHa'],
    hoverinfo='text'
))
fig.update_layout(
    mapbox_style="white-bg", # Ponemos el fondo en blanco
    mapbox_layers=[
        {
            "below": 'traces', # Para que el satélite quede por debajo de tus puntos de rinde
            "sourcetype": "raster",
            "sourceattribution": "Google",
            "source": [
                "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}" # URL del satélite de Google
            ]
        }
    ],
    mapbox_center={"lat": gdf_rinde['Latitud'].mean(), "lon": gdf_rinde['Longitud'].mean()},
    mapbox_zoom=13, # Te subí un poco el zoom (14 o 15 suele ser mejor para ver lotes)
    title="Rinde en el campo"
)
fig.show()

#lo hago html
fig.write_html("mapa_rinde.html")
print("Mapa guardado como mapa_rinde.html")


In [ ]:
#Vamos a crear un .shp con los puntos de rinde para poder abrirlo en QGIS
gdf_rinde.to_file("puntos_rinde.shp")
print("Archivo puntos_rinde.shp creado para QGIS")

NameError: name 'gdf_rinde' is not defined

# Vamos a descargar información satelital para contrastar con el mapa de rendimiento

In [ ]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# Editá estas variables antes de correr el notebook.
# Para mayor seguridad podés usar variables de entorno:
#   import os; GEE_PROJECT = os.environ.get('GEE_PROJECT', 'tu-proyecto')
# ============================================================

GEE_PROJECT   = 'my-project-12126-484118'   # <-- Reemplazá con tu project ID de GEE
CENTRO_LAT   = 33.584              # Latitud del centro del mapa inicial
CENTRO_LON   = -101.845            # Longitud del centro del mapa inicial
ZOOM_INICIAL  = 13
ANIOS_ATRAS   = 3
MAX_NUBES     = 30                  # Umbral de cobertura nubosa (%)
K_EXTINCTION  = 0.5                 # Coef. extinción Beer-Lambert (0.4–0.6 para cultivos)
SAVI_L        = 0.5                 # Factor de corrección de suelo para SAVI (0=suelo desnudo, 1=canopeo cerrado)

print("✅ Configuración cargada.")
print(f"   Proyecto GEE : {GEE_PROJECT}")
print(f"   Centro mapa  : ({CENTRO_LAT}, {CENTRO_LON})")
print(f"   Ventana      : {ANIOS_ATRAS} años | Nubes ≤ {MAX_NUBES}% | k={K_EXTINCTION} | L={SAVI_L}")

✅ Configuración cargada.
   Proyecto GEE : my-project-12126-484118
   Centro mapa  : (33.584, -101.845)
   Ventana      : 3 años | Nubes ≤ 30% | k=0.5 | L=0.5
